In [1]:
import sys
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

In [2]:
from conllu import parse_incr
from steps.projectivize import is_non_proj
import numpy as np
import pandas as pd

In [3]:
def get_non_proj_sentences(gold_path):
    with open(gold_path, "r", encoding="utf-8") as f:
        sent_ids = []
        for tokenlist in parse_incr(f):
            arcs = []
            for token in tokenlist:
                arcs.append((token["head"], token["id"]))
            if is_non_proj(arcs):
                sent_ids.append(tokenlist.metadata["sent_id"])
    return sent_ids

In [4]:
gold_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
non_proj_sents = get_non_proj_sentences(gold_file)

In [104]:
def collect_prediction_stats(gold_path, pred_path):

    stats = {
        "sent_id":[],
        "token_id": [], 
        "token_pos": [],
        "gold_head": [],
        "gold_head_pos":[], 
        "pred_head": [],
        "pred_head_pos":[], 
        "gold_deprel": [],
        "pred_deprel": []
        }    
 
    with open(pred_path, "r", encoding="utf-8") as fpred, \
         open(gold_path, "r", encoding="utf-8") as fgold:
        for sent_pred, sent_gold in zip(parse_incr(fpred), parse_incr(fgold)):
            sent_id = sent_gold.metadata["sent_id"]
            for tok_pred, tok_gold in zip(sent_pred, sent_gold):
                stats["sent_id"].append(sent_id)
                stats["token_id"].append(tok_pred["id"])
                stats["token_pos"].append(tok_gold["upos"])
                stats["gold_head"].append(tok_gold["head"])
                stats["pred_head"].append(tok_pred["head"])
                stats["gold_deprel"].append(tok_gold["deprel"])
                stats["pred_deprel"].append(tok_pred["deprel"])
                stats["gold_head_pos"].append(sent_gold[int(tok_gold["head"])-1]["upos"] if int(tok_gold["head"])-1 != 0 else "ROOT")
                stats["pred_head_pos"].append(sent_pred[int(tok_pred["head"])-1]["upos"] if int(tok_pred["head"])-1 != 0 else "ROOT")

    return stats



## Constituency Parser

In [105]:
gold_const_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
pred_const_file = os.path.join(PROJECT_ROOT, "predictions", "stanza", "lang=en,bert=finetune,charlm=yes,pretrain=yes,epochs=100,deprojz=yes,matched=yes.conllu")
const_stats = collect_prediction_stats(gold_const_file, pred_const_file)

In [106]:
const_df = pd.DataFrame(const_stats)
const_df

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-1,1,ADV,7,PROPN,7,PROPN,discourse,discourse
1,english-penn-test-1,2,PUNCT,7,PROPN,7,PROPN,punct,punct
2,english-penn-test-1,3,PRON,7,PROPN,7,PROPN,nsubj,nsubj
3,english-penn-test-1,4,VERB,7,PROPN,7,PROPN,cop,cop
4,english-penn-test-1,5,PART,7,PROPN,7,PROPN,neg,neg
...,...,...,...,...,...,...,...,...,...
56679,english-penn-test-2416,9,NOUN,5,PROPN,5,PROPN,conj,conj
56680,english-penn-test-2416,10,ADP,11,PROPN,11,PROPN,case,case
56681,english-penn-test-2416,11,PROPN,2,VERB,2,VERB,nmod,nmod
56682,english-penn-test-2416,12,PROPN,2,VERB,2,VERB,nmod:tmod,nmod:tmod


In [107]:
const_wrong_head_df = const_df.query("gold_head != pred_head")
const_wrong_head_df

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
30,english-penn-test-2,23,ADJ,21,NOUN,19,VERB,dep,dep
89,english-penn-test-4,11,PUNCT,5,NOUN,13,NOUN,punct,punct
104,english-penn-test-4,26,PUNCT,5,NOUN,13,NOUN,punct,punct
123,english-penn-test-5,10,NOUN,7,ADP,6,VERB,nmod,nmod
333,english-penn-test-15,12,ADV,13,DET,14,NOUN,advmod,advmod
...,...,...,...,...,...,...,...,...,...
56555,english-penn-test-2409,32,PUNCT,8,VERB,26,VERB,punct,punct
56614,english-penn-test-2413,14,NOUN,4,VERB,6,NOUN,nmod,nmod
56639,english-penn-test-2414,16,PUNCT,3,VERB,8,VERB,punct,punct
56640,english-penn-test-2414,17,VERB,20,NOUN,8,VERB,case,xcomp


## MST Parser

In [108]:
gold_dep_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
pred_dep_file = os.path.join(PROJECT_ROOT, "predictions", "depparse", "lang=en,bert=frozen,charlm=no,pretrain=no.conllu")
mst_stats = collect_prediction_stats(gold_dep_file, pred_dep_file)

In [92]:
mst_df = pd.DataFrame(mst_stats)
mst_df

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-1,1,ADV,7,PROPN,7,PROPN,discourse,discourse
1,english-penn-test-1,2,PUNCT,7,PROPN,7,PROPN,punct,punct
2,english-penn-test-1,3,PRON,7,PROPN,7,PROPN,nsubj,nsubj
3,english-penn-test-1,4,VERB,7,PROPN,7,PROPN,cop,cop
4,english-penn-test-1,5,PART,7,PROPN,7,PROPN,neg,neg
...,...,...,...,...,...,...,...,...,...
56679,english-penn-test-2416,9,NOUN,5,PROPN,5,PROPN,conj,conj
56680,english-penn-test-2416,10,ADP,11,PROPN,11,PROPN,case,case
56681,english-penn-test-2416,11,PROPN,2,VERB,2,VERB,nmod,nmod
56682,english-penn-test-2416,12,PROPN,2,VERB,2,VERB,nmod:tmod,nmod:tmod


In [93]:
mst_wrong_head_df = mst_df.query("gold_head != pred_head")
mst_wrong_head_df 

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
29,english-penn-test-2,22,PUNCT,23,ADJ,19,ADJ,punct,punct
30,english-penn-test-2,23,ADJ,21,NOUN,19,NOUN,dep,nsubj
36,english-penn-test-2,29,NOUN,23,ADJ,19,ADJ,nmod,nmod
37,english-penn-test-2,30,PUNCT,23,ADJ,33,ADJ,punct,punct
59,english-penn-test-3,12,VERB,0,PUNCT,18,PUNCT,root,advcl
...,...,...,...,...,...,...,...,...,...
56555,english-penn-test-2409,32,PUNCT,8,VERB,26,VERB,punct,punct
56614,english-penn-test-2413,14,NOUN,4,VERB,6,VERB,nmod,nmod
56639,english-penn-test-2414,16,PUNCT,3,VERB,8,VERB,punct,punct
56640,english-penn-test-2414,17,VERB,20,NOUN,8,NOUN,case,xcomp


## Permuatation

In [94]:
def permutation_test(x, y, n_perm=10000):
    observed = np.mean(x) - np.mean(y)
    combined = np.concatenate([x, y])
    count = 0

    for _ in range(n_perm):
        np.random.shuffle(combined)
        new_x = combined[:len(x)]
        new_y = combined[len(x):]
        if abs(np.mean(new_x) - np.mean(new_y)) >= abs(observed):
            count += 1

    return observed, count / n_perm

In [95]:
obs, p = permutation_test(mst_arc_dist_df["arc_dist"], const_arc_dist_df["arc_dist"])
print("Observation value:", obs)
print("P-value:", p)

Observation value: 0.5677701501502357
P-value: 0.0007


## Find Samples

In [98]:
mst_wrong_arcs = set(mst_wrong_head_df.index)
len(mst_wrong_arcs)

3590

In [99]:
const_wrong_arcs = set(const_wrong_head_df.index)
len(const_wrong_arcs)

1957

In [100]:
intersection = mst_wrong_arcs & const_wrong_arcs
len(intersection)

1382

In [109]:
mst_only = mst_wrong_head_df.loc[list(mst_wrong_arcs - intersection), :]
mst_only.query("pred_head_pos != gold_head_pos")

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
49241,english-penn-test-2085,13,PUNCT,1,ROOT,9,SYM,punct,punct
49270,english-penn-test-2088,8,PROPN,1,ROOT,3,PROPN,nmod,nmod
49305,english-penn-test-2091,11,PUNCT,1,ROOT,7,SYM,punct,punct
49320,english-penn-test-2093,5,NOUN,10,NOUN,1,ROOT,compound,nmod
49325,english-penn-test-2093,10,NOUN,1,ROOT,5,SYM,nmod,conj
...,...,...,...,...,...,...,...,...,...
56475,english-penn-test-2404,8,VERB,1,ROOT,7,SYM,acl,acl
16292,english-penn-test-681,8,PUNCT,1,ROOT,5,PROPN,punct,punct
49081,english-penn-test-2073,14,PUNCT,1,ROOT,10,SYM,punct,punct
49089,english-penn-test-2074,4,PUNCT,1,ROOT,3,PROPN,punct,punct


In [112]:
const_only = const_wrong_head_df.loc[list(const_wrong_arcs - intersection), :]

In [ ]:
# TODO: Const tends to pick a token with different POS, check if it's because of deprojectivization by sort the non-projective arcs out
const_only.query("pred_head_pos != gold_head_pos")

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
32769,english-penn-test-1319,6,PUNCT,7,ADJ,3,PROPN,punct,punct
40971,english-penn-test-1677,12,ADV,13,NUM,14,NOUN,advmod,advmod
32781,english-penn-test-1319,18,ADV,17,VERB,19,ADV,advmod,advmod
53267,english-penn-test-2262,17,VERB,15,NOUN,10,VERB,dep,dep
22571,english-penn-test-916,14,PROPN,12,PROPN,18,NOUN,conj,dummy
...,...,...,...,...,...,...,...,...,...
2018,english-penn-test-88,22,CONJ,15,PROPN,29,ADV,cc,cc
49120,english-penn-test-2077,7,PUNCT,6,NOUN,2,NUM,punct,punct
49123,english-penn-test-2077,10,PROPN,6,NOUN,2,NUM,appos,appos
2023,english-penn-test-88,27,PROPN,15,PROPN,29,ADV,conj,conj


In [119]:
mst_only.query("pred_head_pos != gold_head_pos")

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
49241,english-penn-test-2085,13,PUNCT,1,ROOT,9,SYM,punct,punct
49270,english-penn-test-2088,8,PROPN,1,ROOT,3,PROPN,nmod,nmod
49305,english-penn-test-2091,11,PUNCT,1,ROOT,7,SYM,punct,punct
49320,english-penn-test-2093,5,NOUN,10,NOUN,1,ROOT,compound,nmod
49325,english-penn-test-2093,10,NOUN,1,ROOT,5,SYM,nmod,conj
...,...,...,...,...,...,...,...,...,...
56475,english-penn-test-2404,8,VERB,1,ROOT,7,SYM,acl,acl
16292,english-penn-test-681,8,PUNCT,1,ROOT,5,PROPN,punct,punct
49081,english-penn-test-2073,14,PUNCT,1,ROOT,10,SYM,punct,punct
49089,english-penn-test-2074,4,PUNCT,1,ROOT,3,PROPN,punct,punct


In [123]:
const_only.query("pred_head_pos != gold_head_pos").groupby(["gold_head_pos", "pred_head_pos"])["token_id"].count().sort_values(ascending=False)

gold_head_pos  pred_head_pos
NOUN           VERB             72
VERB           NOUN             54
NOUN           ADJ              22
PROPN          NOUN             18
NOUN           PROPN            15
                                ..
SYM            PUNCT             1
VERB           ADP               1
SYM            NOUN              1
VERB           ROOT              1
               X                 1
Name: token_id, Length: 61, dtype: int64

In [124]:
mst_only.query("pred_head_pos != gold_head_pos").groupby(["gold_head_pos", "pred_head_pos"])["token_id"].count().sort_values(ascending=False)

gold_head_pos  pred_head_pos
ROOT           SYM              11
NOUN           ROOT             10
ROOT           PROPN             9
               DET               5
VERB           ROOT              5
ROOT           VERB              5
ADJ            ROOT              4
ROOT           NOUN              4
               NUM               3
PROPN          ROOT              2
PUNCT          ROOT              2
ROOT           ADJ               1
Name: token_id, dtype: int64

In [ ]:
mst_wrong_head_df.query("gold_head_pos == 'ROOT' and token_pos == 'VERB'")

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
31744,english-penn-test-1278,15,VERB,1,ROOT,6,DET,acl:relcl,acl:relcl
36314,english-penn-test-1473,13,VERB,1,ROOT,8,NOUN,conj,conj
43673,english-penn-test-1816,28,VERB,1,ROOT,0,ADJ,parataxis,root
45238,english-penn-test-1890,2,VERB,1,ROOT,0,NOUN,acl,root
45438,english-penn-test-1905,5,VERB,1,ROOT,4,NOUN,acl,acl
56223,english-penn-test-2392,15,VERB,1,ROOT,0,SYM,acl,root
56310,english-penn-test-2396,14,VERB,1,ROOT,0,SYM,acl,root
56383,english-penn-test-2400,8,VERB,1,ROOT,7,SYM,acl,acl
56475,english-penn-test-2404,8,VERB,1,ROOT,7,SYM,acl,acl


In [128]:
const_wrong_head_df.query("gold_head_pos == 'ROOT' and token_pos == 'VERB'")

,sent_id,token_id,token_pos,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
43673,english-penn-test-1816,28,VERB,1,ROOT,3,PRON,parataxis,dep
45438,english-penn-test-1905,5,VERB,1,ROOT,4,NOUN,acl,acl
56310,english-penn-test-2396,14,VERB,1,ROOT,7,NOUN,acl,acl


## Bucket groups error rates

In [ ]:
dist_1_to_2 = const_df.query("abs(gold_head - token_id) <= 2")["token_id"].count()
dist_1_to_2

np.int64(32971)

In [ ]:
dist_3_to_5 = const_df.query("3 <= abs(gold_head - token_id) <= 5")["token_id"].count()
dist_3_to_5

np.int64(13921)

In [ ]:
dist_6_to_10 = const_df.query("6 <= abs(gold_head - token_id) <= 10")["token_id"].count()
dist_6_to_10

np.int64(5281)

In [ ]:
dist_11_plus = const_df.query("abs(gold_head - token_id) >= 11")["token_id"].count()
dist_11_plus

np.int64(4511)

In [ ]:
const_1_to_2 = const_wrong_head_df.query("abs(gold_head - token_id) <= 2")["token_id"].count()
const_1_to_2 

np.int64(718)

In [ ]:
const_3_to_5 = const_wrong_head_df.query("3 <= abs(gold_head - token_id) <= 5")["token_id"].count()
const_3_to_5

np.int64(567)

In [ ]:
const_6_to_10 = const_wrong_head_df.query("6 <= abs(gold_head - token_id) <= 10")["token_id"].count()
const_6_to_10

np.int64(394)

In [ ]:
const_11_plus = const_wrong_head_df.query("abs(gold_head - token_id) >= 11")["token_id"].count()
const_11_plus

np.int64(278)

In [ ]:
mst_1_to_2 = mst_wrong_head_df.query("abs(gold_head - token_id) <= 2")["token_id"].count()
mst_1_to_2 

np.int64(1204)

In [ ]:
mst_3_to_5 = mst_wrong_head_df.query("3 <= abs(gold_head - token_id) <= 5")["token_id"].count()
mst_3_to_5

np.int64(981)

In [ ]:
mst_6_to_10 = mst_wrong_head_df.query("6 <= abs(gold_head - token_id) <= 10")["token_id"].count()
mst_6_to_10

np.int64(785)

In [ ]:
mst_11_plus = mst_wrong_head_df.query("abs(gold_head - token_id) >= 11")["token_id"].count()
mst_11_plus

np.int64(620)

In [ ]:
print(f"Distance 1-2: Constitueny={const_1_to_2/dist_1_to_2*100:.2f}, Dependency={mst_1_to_2/dist_1_to_2*100:.2f}")
print(f"Distance 3-5: Constitueny={const_3_to_5/dist_3_to_5*100:.2f}, Dependency={mst_3_to_5/dist_3_to_5*100:.2f}")
print(f"Distance 6-10: Constitueny={const_6_to_10/dist_6_to_10*100:.2f}, Dependency={mst_6_to_10/dist_6_to_10*100:.2f}")
print(f"Distance 11: Constitueny={const_11_plus/dist_11_plus*100:.2f}, Dependency={mst_11_plus/dist_11_plus*100:.2f}")

Distance 1-2: Constitueny=2.18, Dependency=3.65
Distance 3-5: Constitueny=4.07, Dependency=7.05
Distance 6-10: Constitueny=7.46, Dependency=14.86
Distance 11: Constitueny=6.16, Dependency=13.74


In [1]:
import stanza
stanza.download("zh-hans")

c:\Users\hrkwl\.conda\envs\projz\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-07 14:16:01 INFO: Downloaded file to C:\Users\hrkwl\stanza_resources\resources.json
2026-01-07 14:16:01 INFO: Downloading default packages for language: zh-hans (Simplified_Chinese) ...
2026-01-07 14:17:47 INFO: Downloaded file to C:\Users\hrkwl\stanza_resources\zh-hans\default.zip
2026-01-07 14:17:56 INFO: Finished downloading models and saved to C:\Users\hrkwl\stanza_resources
